# browsergraph — composable browser automation

Write a graph **once**; run it on Playwright, Patchright, Selenium,
undetected-chromedriver or Camoufox without changing a line.

This notebook runs entirely inside Kaggle: **no browser and no network are
required**. The core is stdlib-only, and the `mock` engine plus a local HTTP
server exercise the real code paths.

> Kaggle kernels have no Ollama, so the LLM sections use a stub client. The
> same code runs against a real model by setting `OLLAMA_HOST`.


## 1. Install


In [ ]:
%pip install -q browsergraph

import browsergraph
print('browsergraph', browsergraph.__version__)


## 2. What can this machine run?

`doctor` reports what is present and prints the fix for anything missing —
browser automation fails for environmental reasons far more often than
logical ones.


In [ ]:
from browsergraph.doctor import run_all
print(run_all().text())


## 3. A graph is nodes over a dimension space

`Spec` is one point in that space: engine, binary, transport, display,
stealth, preprocessing, vision, behaviour. Nodes never touch an engine
directly — they talk to a `BrowserPort`, which is what makes one graph run
everywhere.


In [ ]:
from browsergraph import Graph, Spec, Engine, run
from browsergraph.nodes.actions import Navigate, WaitFor, Click, Extract
from browsergraph.drivers.mock import MockBrowser

PAGES = {'https://acme.example': {'h1': 'Acme Roofing', '#login': 'Log in'}}

graph = (Graph('demo')
         .add(Navigate('https://acme.example'))
         .add(WaitFor('#login'))
         .add(Extract('h1', into='heading')))

result = run(graph, Spec(engine=Engine.MOCK), MockBrowser(pages=PAGES))
print(result.summary())
print('heading:', result.context.data['heading'])


## 4. The linter catches the failure that matters most

**BG003: a graph that changes remote state but never verifies the outcome.**

This rule exists because of a real incident: 551 emails reported "sent"
successfully and produced zero posts. Every layer said success; nothing
checked the destination.


In [ ]:
from browsergraph.lint import lint, report

risky = (Graph('risky')
         .add(Navigate('https://acme.example'))
         .add(WaitFor('#login'))
         .add(Click('#login')))          # mutates, never verifies

print(report(lint(risky)))


## 5. Token reduction: preprocessing + focus

Raw HTML for a modern page is mostly framework noise. Eight preprocessing
strategies trade structure against size; `focus` then keeps only the chunks
that answer the question — plus their **neighbours**, because the chunk that
matches "contact" is rarely the one holding the phone number.


In [ ]:
from browsergraph.preprocess import Preprocess, compare, reduce
from browsergraph.focus import focus

HTML = '''<!doctype html><html><head><title>Acme</title>
<style>.a{color:red}</style><script>var big="''' + 'z'*4000 + '''";</script></head>
<body><nav><a href="/">Home</a></nav><main><h1>Contact Acme</h1>
<p>''' + 'Filler about the company. '*80 + '''</p>
<h2>Sales</h2><p>Email sales@acme.example or call (303) 555-0142.</p>
<button id=send data-testid=send>Send</button></main>
<footer>(c) Acme</footer></body></html>'''

for r in sorted(compare(HTML), key=lambda r: r.chars):
    print(f'{r.strategy.value:<14} {r.chars:>7} chars  saved {r.saved_pct:5.1f}%')

md_view = reduce(HTML, Preprocess.MARKDOWN).content
f = focus(md_view, 'sales email address', budget=500)
print(f'\nfocused: {f.chars} chars, saved {f.saved_pct:.0f}%')
print('answer retained:', 'sales@acme.example' in f.content)


## 6. Deterministic extraction

No model involved. These run on every page, so they are conservative by
design: **a false positive silently poisons a dataset, a miss is visible as
an empty field.** Dates, repeated digits and asset filenames are rejected.


In [ ]:
from browsergraph.extract.patterns import extract_contacts
from browsergraph.extract.content import text_of

found = extract_contacts(text_of(HTML), [])
print('emails :', found.emails)
print('phones :', [p.raw for p in found.phones])

# things that look like contacts but are not
noise = extract_contacts('Order 2026-03-04, SKU 000000000, logo@2x.png', [])
print('\nnoise rejected ->', noise.to_dict()['emails'], noise.to_dict()['phones'])


## 7. NAICS classification, with honest confidence

A weak match reports `usable=False` rather than emitting a plausible-looking
code. Silently classifying every input is the failure this avoids.


In [ ]:
from browsergraph.classify.naics import classify

for label, text in [
    ('roofing firm', 'We are a general contractor specialising in roofing and gutters.'),
    ('restaurant',   'Our restaurant menu, dining, reservations and catering.'),
    ('empty page',   'Welcome to our website. Hello.'),
]:
    c = classify(text)
    print(f'{label:<14} {c.code or "--":<6} {c.confidence:<7} usable={c.usable}')


## 8. The dimension space, and why you should not enumerate it

Incompatible combinations are rejected **with reasons**. Full enumeration
explodes, so `sample` builds a pairwise covering array — most failures are
two-value interactions.


In [ ]:
from browsergraph.combos import count, rejected
from browsergraph.sample import coverage, sample_specs
from browsergraph.dimensions import Binary, Display, Stealth

total, ok = count()
print(f'{total} combinations -> {ok} runnable, {total-ok} rejected\n')
for desc, why in rejected()[:3]:
    print(f'  {desc}\n      {why[0]}')

axes = {'engine': list(Engine), 'binary': list(Binary),
        'display': list(Display), 'stealth': list(Stealth)}
specs = sample_specs(axes)
cov, poss = coverage(axes, specs)
print(f'\npairwise: {len(specs)} runs cover {cov}/{poss} value-pairs')


## 9. Self-tuning: learn from similar sites

Rather than re-deriving the right configuration each time, outcomes are
recorded and generalised: `site -> org -> sector -> platform -> global`.

**Evidence is reported honestly.** One success is `p≈0.67, n=1` after
smoothing — not certainty.


In [ ]:
from browsergraph.learn import Features, Knowledge, Outcome, plan
from browsergraph.strategy import ladder

k = Knowledge()
winner = Spec(engine=Engine.MOCK)

f1 = Features.of('https://acme.example', task='contacts')
k.record(f1, winner, True)
print('after one win :', k.estimate(f1, winner.describe()))

# a brand-new site in the same sector inherits useful priors
for i in range(6):
    k.record(Features.of(f'https://shop{i}.example', sector='44-45',
                         task='contacts'), winner, True)
fresh = Features.of('https://brandnew.example', sector='44-45', task='contacts')
print('unseen site   :', k.estimate(fresh, winner.describe()))

p = plan(k, f1, ladder(Spec(engine=Engine.MOCK)))
print('\n' + p.explain())


## 10. Guardrails: stop early, but never on ignorance

A budget caps attempts, time and model calls. Early stopping requires **both**
low expected success and enough evidence to trust that number — abandoning an
unfamiliar site on its first failure is exactly when exploration is worth most.


In [ ]:
from browsergraph.learn import Budget, Estimate

b = Budget(min_expected_success=0.5, min_evidence_to_stop=3)
print('thin evidence  ->', repr(b.should_stop_early(Estimate('s', p=0.1, evidence=1.0))))
print('solid evidence ->', b.should_stop_early(Estimate('s', p=0.1, evidence=8.0)))


## 11. Failure classification: a CAPTCHA is not a missing element

Retrying is not a universal remedy. A bot wall must **abort**, not retry —
that is how accounts get banned.


In [ ]:
from browsergraph.errors import classify as classify_error

for err, page in [('click target not found: #login', ''),
                  ('element not found: #x', 'Please complete the CAPTCHA'),
                  ('HTTP 429 too many requests', ''),
                  ('403 Forbidden', '')]:
    d = classify_error(err, page)
    print(f'{d.failure.value:<14} -> {d.response.value:<11} terminal={d.terminal}')


## 12. Tasks: say *what*, not *how*

Six built-in capabilities, each with declared parameters so a bad request
fails at submission rather than on page 400.


In [ ]:
from browsergraph.tasks import catalog
for t in catalog():
    print(f"{t['name']:<13} {t['summary'][:64]}")


## 13. Running a real task against a real browser

Kaggle has no browser, so this cell is illustrative. On a machine with
Playwright installed it runs unchanged.


In [ ]:
# pip install browsergraph[playwright] && playwright install chromium
#
# from browsergraph.tasks import make
# from browsergraph.drivers import build
#
# spec = Spec(engine=Engine.PLAYWRIGHT)
# result = make('research', url='https://example.com').run(build(spec))
# print(result.to_dict())
print('see the README for live examples')


## 14. Multi-model routing

Selection is by **reported capability**, not by name. A vision job answered
by a text model returns confident fiction, so a model that lacks the
capability raises rather than silently substituting.


In [ ]:
from browsergraph.routing import JOBS
print('jobs and what they require:')
for job, spec in JOBS.items():
    print(f"  {job:<10} needs={spec['capability']:<11} prefers={spec['role'] or '-'}")

# with a reachable Ollama:
# from browsergraph.routing import plan_assignments
# print(plan_assignments())


---

## Where to go next

- **Repo**: <https://github.com/Amarel-Taylor-Scott/browsergraph>
- `ARCHITECTURE.md` — the Protocol-vs-base-class seam that makes engines interchangeable
- `DIMENSIONS.md` — axes still worth adding, and why verification matters most
- `ISOLATION.md` — running conflicting engines in separate virtualenvs
- `PLUGINS.md` — the open plugin format

MIT licensed.
